# ⚡ Elektroniker EFZ – Kapitel: Widerstand
**Zusammenfassung mit integrierten Rechnern**

---

In [ ]:
# ============================================================
# SETUP – Bitte zuerst ausführen!
# ============================================================
import re
import ipywidgets as widgets
from IPython.display import display, HTML
import math

# --- SI-Präfix Parser ---
SI_PREFIXES = {
    'p': 1e-12, 'n': 1e-9,
    'u': 1e-6,  'µ': 1e-6,
    'm': 1e-3,  '':  1.0,
    'k': 1e3,   'M': 1e6,  'G': 1e9
}

def p(val_str):
    """Parst Zahlen mit optionalem SI-Präfix: z.B. 10k, 4.7u, 100n"""
    if val_str is None: return None
    val_str = val_str.strip()
    if val_str == "": return None
    match = re.fullmatch(r'([-+]?\d*\.?\d+)\s*([pnumkMGµ]?)', val_str)
    if not match:
        raise ValueError(f"Ungültiger Wert: '{val_str}'")
    value, prefix = match.groups()
    return float(value) * SI_PREFIXES[prefix]

def fmt(val, unit=''):
    """Formatiert Zahlen schön mit SI-Präfix."""
    if val == 0:
        return f"0 {unit}"
    abs_val = abs(val)
    prefixes = [('G', 1e9), ('M', 1e6), ('k', 1e3), ('', 1),
                ('m', 1e-3), ('µ', 1e-6), ('n', 1e-9), ('p', 1e-12)]
    for name, factor in prefixes:
        if abs_val >= factor:
            return f"{val/factor:.4g} {name}{unit}"
    return f"{val:.4e} {unit}"

def make_input(description, placeholder='leer lassen = berechnen'):
    return widgets.Text(
        description=description,
        placeholder=placeholder,
        layout=widgets.Layout(width='320px'),
        style={'description_width': '130px'}
    )

def make_button(label='🔢 Berechnen'):
    return widgets.Button(
        description=label,
        button_style='primary',
        layout=widgets.Layout(width='180px', margin='8px 0')
    )

---
## 1. 📖 Grundlagen des Widerstands

Der elektrische **Widerstand** $R$ gibt an, wie stark ein Bauteil den Stromfluss hemmt.

| Grösse | Symbol | Einheit |
|--------|--------|---------|
| Widerstand | $R$ | Ohm [Ω] |
| Spannung | $U$ | Volt [V] |
| Strom | $I$ | Ampere [A] |
| Leistung | $P$ | Watt [W] |

### Ohmsches Gesetz
$$U = R \cdot I$$

> **Merkhilfe:** Das **URI-Dreieck** – decke ab, was du suchen willst:
> ```
>     [U]
>    ------
>   [R] | [I]
> ```

In [ ]:
# ============================================================
# RECHNER 1: Ohmsches Gesetz  U = R · I
# ============================================================
display(HTML("<h3>🔢 Rechner 1 – Ohmsches Gesetz</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

U_ohm = make_input('U [V]  =')
R_ohm = make_input('R [Ω]  =')
I_ohm = make_input('I [A]  =')
btn_ohm = make_button()
out_ohm = widgets.Output()

def calc_ohm(_=None):
    out_ohm.clear_output()
    with out_ohm:
        try:
            U = p(U_ohm.value)
            R = p(R_ohm.value)
            I = p(I_ohm.value)
            vals = {'U': U, 'R': R, 'I': I}
            none_fields = [k for k, v in vals.items() if v is None]
            if len(none_fields) == 0:
                print("❌ Ein Feld leer lassen!")
                return
            if len(none_fields) > 1:
                print("❌ Nur ein Feld leer lassen!")
                return
            miss = none_fields[0]
            if miss == 'U': res = R * I;   label = 'U'
            elif miss == 'R': res = U / I; label = 'R'
            else:             res = U / R; label = 'I'
            units = {'U': 'V', 'R': 'Ω', 'I': 'A'}
            print(f"✅ {label} = {fmt(res, units[label])}")
            print(f"   ({res:.6e} {units[label]})")
        except Exception as e:
            print("❌ Fehler:", e)

btn_ohm.on_click(calc_ohm)
for field in [U_ohm, R_ohm, I_ohm]:
    field.observe(calc_ohm, names='value')

display(widgets.VBox([U_ohm, R_ohm, I_ohm, btn_ohm, out_ohm]))

---
## 2. ⚡ Leistung und Energie

$$P = U \cdot I = \frac{U^2}{R} = I^2 \cdot R$$

| Formel | Verwendung |
|--------|------------|
| $P = U \cdot I$ | wenn U und I bekannt |
| $P = I^2 \cdot R$ | wenn I und R bekannt |
| $P = U^2 / R$ | wenn U und R bekannt |

**Energie:** $W = P \cdot t$ \[Joule = Watt · Sekunden\]

In [ ]:
# ============================================================
# RECHNER 2: Leistung  P = U · I
# ============================================================
display(HTML("<h3>🔢 Rechner 2 – Elektrische Leistung</h3>"
             "<i>Ein Feld leer lassen → wird berechnet. Für P=I²·R oder P=U²/R: P + R + {I oder U}</i>"))

P_inp = make_input('P [W]  =')
U_inp = make_input('U [V]  =')
I_inp = make_input('I [A]  =')
R_inp = make_input('R [Ω]  =')
btn_P = make_button()
out_P = widgets.Output()

def calc_P(_=None):
    out_P.clear_output()
    with out_P:
        try:
            PP = p(P_inp.value)
            UU = p(U_inp.value)
            II = p(I_inp.value)
            RR = p(R_inp.value)

            # P = U·I
            if PP is None and UU is not None and II is not None:
                print(f"✅ P = U·I = {fmt(UU*II, 'W')}")
            elif UU is None and PP is not None and II is not None:
                print(f"✅ U = P/I = {fmt(PP/II, 'V')}")
            elif II is None and PP is not None and UU is not None:
                print(f"✅ I = P/U = {fmt(PP/UU, 'A')}")
            # P = I²·R
            elif PP is None and II is not None and RR is not None:
                print(f"✅ P = I²·R = {fmt(II**2 * RR, 'W')}")
            elif RR is None and PP is not None and II is not None:
                print(f"✅ R = P/I² = {fmt(PP/II**2, 'Ω')}")
            elif II is None and PP is not None and RR is not None:
                print(f"✅ I = √(P/R) = {fmt(math.sqrt(PP/RR), 'A')}")
            # P = U²/R
            elif PP is None and UU is not None and RR is not None:
                print(f"✅ P = U²/R = {fmt(UU**2 / RR, 'W')}")
            elif RR is None and PP is not None and UU is not None:
                print(f"✅ R = U²/P = {fmt(UU**2 / PP, 'Ω')}")
            elif UU is None and PP is not None and RR is not None:
                print(f"✅ U = √(P·R) = {fmt(math.sqrt(PP*RR), 'V')}")
            else:
                print("ℹ️ Gib genau 2 Werte ein (z.B. U + I, oder I + R, oder U + R)")
        except Exception as e:
            print("❌ Fehler:", e)

btn_P.on_click(calc_P)
for field in [P_inp, U_inp, I_inp, R_inp]:
    field.observe(calc_P, names='value')

display(widgets.VBox([P_inp, U_inp, I_inp, R_inp, btn_P, out_P]))

---
## 3. 🔗 Reihenschaltung von Widerständen

$$R_{ges} = R_1 + R_2 + R_3 + \ldots$$

**Eigenschaften:**
- Überall **gleicher Strom**: $I_{ges} = I_1 = I_2 = I_3$
- Spannungen addieren sich: $U_{ges} = U_1 + U_2 + U_3$
- Gesamtwiderstand **grösser** als jeder Einzelwiderstand

**Spannungsteiler:**
$$U_x = U_{ges} \cdot \frac{R_x}{R_{ges}}$$

In [ ]:
# ============================================================
# RECHNER 3: Reihenschaltung (bis 5 Widerstände)
# ============================================================
display(HTML("<h3>🔢 Rechner 3 – Reihenschaltung</h3>"
             "<i>Bis zu 5 Widerstände. Leere Felder werden ignoriert. Mindestens 2 Werte.</i>"))

r_fields = [make_input(f'R{i+1} [Ω] =') for i in range(5)]
U_reihe = make_input('U_ges [V] =')
btn_reihe = make_button()
out_reihe = widgets.Output()

def calc_reihe(_=None):
    out_reihe.clear_output()
    with out_reihe:
        try:
            Rs = [p(f.value) for f in r_fields]
            Rs_valid = [r for r in Rs if r is not None]
            U_ges = p(U_reihe.value)
            if len(Rs_valid) < 1:
                print("ℹ️ Mindestens 1 Widerstand eingeben")
                return
            R_ges = sum(Rs_valid)
            print(f"✅ R_ges = {fmt(R_ges, 'Ω')}  ({R_ges:.6e} Ω)")
            if U_ges is not None and R_ges > 0:
                I = U_ges / R_ges
                print(f"   I     = {fmt(I, 'A')}  (bei U_ges = {fmt(U_ges, 'V')})")
                for i, r in enumerate(Rs):
                    if r is not None:
                        U_teil = I * r
                        print(f"   U_R{i+1}  = {fmt(U_teil, 'V')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_reihe.on_click(calc_reihe)
for f in r_fields + [U_reihe]:
    f.observe(calc_reihe, names='value')

display(widgets.VBox(r_fields + [U_reihe, btn_reihe, out_reihe]))

---
## 4. ⚡ Parallelschaltung von Widerständen

$$\frac{1}{R_{ges}} = \frac{1}{R_1} + \frac{1}{R_2} + \frac{1}{R_3} + \ldots$$

**Für 2 Widerstände (Produktformel):**
$$R_{ges} = \frac{R_1 \cdot R_2}{R_1 + R_2}$$

**Eigenschaften:**
- Überall **gleiche Spannung**: $U_{ges} = U_1 = U_2 = U_3$
- Ströme addieren sich: $I_{ges} = I_1 + I_2 + I_3$
- Gesamtwiderstand **kleiner** als der kleinste Einzelwiderstand

**Stromteiler (2 Widerstände):**
$$I_1 = I_{ges} \cdot \frac{R_2}{R_1 + R_2}$$

In [ ]:
# ============================================================
# RECHNER 4: Parallelschaltung (bis 5 Widerstände)
# ============================================================
display(HTML("<h3>🔢 Rechner 4 – Parallelschaltung</h3>"
             "<i>Bis zu 5 Widerstände. Leere Felder werden ignoriert.</i>"))

rp_fields = [make_input(f'R{i+1} [Ω] =') for i in range(5)]
U_para = make_input('U_ges [V] =')
btn_para = make_button()
out_para = widgets.Output()

def calc_para(_=None):
    out_para.clear_output()
    with out_para:
        try:
            Rs = [p(f.value) for f in rp_fields]
            Rs_valid = [r for r in Rs if r is not None]
            U_ges = p(U_para.value)
            if len(Rs_valid) < 1:
                print("ℹ️ Mindestens 1 Widerstand eingeben")
                return
            R_ges = 1 / sum(1/r for r in Rs_valid)
            print(f"✅ R_ges = {fmt(R_ges, 'Ω')}  ({R_ges:.6e} Ω)")
            if U_ges is not None:
                I_ges = U_ges / R_ges
                print(f"   I_ges = {fmt(I_ges, 'A')}  (bei U_ges = {fmt(U_ges, 'V')})")
                for i, r in enumerate(Rs):
                    if r is not None:
                        I_teil = U_ges / r
                        print(f"   I_R{i+1}  = {fmt(I_teil, 'A')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_para.on_click(calc_para)
for f in rp_fields + [U_para]:
    f.observe(calc_para, names='value')

display(widgets.VBox(rp_fields + [U_para, btn_para, out_para]))

---
## 5. 🌡️ Temperaturabhängigkeit des Widerstands

$$R_T = R_{20} \cdot [1 + \alpha_{20} \cdot (T - 20°C)]$$

| Material | $\alpha_{20}$ [1/K] |
|----------|---------------------|
| Kupfer (Cu) | 0.00393 |
| Aluminium (Al) | 0.00403 |
| Eisen (Fe) | 0.00650 |
| Konstantan | 0.00001 |
| Silber (Ag) | 0.00380 |

- **$\alpha_{20}$** = Temperaturkoeffizient (bei 20°C Referenz)
- Metalle: $\alpha > 0$ → Widerstand steigt mit Temperatur (PTC)
- NTC-Widerstände: $\alpha < 0$ → Widerstand sinkt mit Temperatur

In [ ]:
# ============================================================
# RECHNER 5: Temperaturabhängigkeit  R_T = R_20 · [1 + α·(T-20)]
# ============================================================
display(HTML("<h3>🔢 Rechner 5 – Temperaturabhängigkeit</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

R20_inp  = make_input('R₂₀ [Ω]  =')
alpha_inp = make_input('α₂₀ [1/K] =', 'z.B. 0.00393 für Cu')
T_inp    = make_input('T [°C]   =')
RT_inp   = make_input('R_T [Ω]  =')

# Materialvorauswahl
mat_select = widgets.Dropdown(
    options=[('-- Material wählen --', None),
             ('Kupfer Cu', 0.00393), ('Aluminium Al', 0.00403),
             ('Eisen Fe', 0.00650), ('Konstantan', 0.00001), ('Silber Ag', 0.00380)],
    description='Material:', style={'description_width': '130px'},
    layout=widgets.Layout(width='320px')
)

def set_alpha(change):
    if change['new'] is not None:
        alpha_inp.value = str(change['new'])
mat_select.observe(set_alpha, names='value')

btn_temp = make_button()
out_temp = widgets.Output()

def calc_temp(_=None):
    out_temp.clear_output()
    with out_temp:
        try:
            R20 = p(R20_inp.value)
            alpha = p(alpha_inp.value)
            T = p(T_inp.value)
            RT = p(RT_inp.value)
            vals = {'R₂₀': R20, 'α₂₀': alpha, 'T': T, 'R_T': RT}
            none_fields = [k for k, v in vals.items() if v is None]
            if len(none_fields) == 0:
                print("❌ Ein Feld leer lassen!")
                return
            if len(none_fields) > 1:
                print("❌ Nur ein Feld leer lassen!")
                return
            miss = none_fields[0]
            if miss == 'R_T':
                res = R20 * (1 + alpha * (T - 20))
                print(f"✅ R_T  = {fmt(res, 'Ω')}")
            elif miss == 'R₂₀':
                res = RT / (1 + alpha * (T - 20))
                print(f"✅ R₂₀  = {fmt(res, 'Ω')}")
            elif miss == 'T':
                res = ((RT / R20) - 1) / alpha + 20
                print(f"✅ T    = {res:.4g} °C")
            elif miss == 'α₂₀':
                res = ((RT / R20) - 1) / (T - 20)
                print(f"✅ α₂₀  = {res:.6g} 1/K")
        except Exception as e:
            print("❌ Fehler:", e)

btn_temp.on_click(calc_temp)
for f in [R20_inp, alpha_inp, T_inp, RT_inp]:
    f.observe(calc_temp, names='value')

display(widgets.VBox([mat_select, R20_inp, alpha_inp, T_inp, RT_inp, btn_temp, out_temp]))

---
## 6. 📏 Spezifischer Widerstand / Leitungswiderstand

$$R = \rho \cdot \frac{l}{A}$$

| Grösse | Symbol | Einheit |
|--------|--------|---------|
| Widerstand | $R$ | Ω |
| Spez. Widerstand | $\rho$ (rho) | Ω·mm²/m |
| Länge | $l$ | m |
| Querschnitt | $A$ | mm² |

| Material | ρ [Ω·mm²/m] |
|----------|-------------|
| Kupfer Cu | 0.0178 |
| Aluminium Al | 0.0283 |
| Eisen Fe | 0.100 |
| Konstantan | 0.500 |

In [ ]:
# ============================================================
# RECHNER 6: Leitungswiderstand  R = ρ · l / A
# ============================================================
display(HTML("<h3>🔢 Rechner 6 – Leitungswiderstand</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

rho_select = widgets.Dropdown(
    options=[('-- Material wählen --', None),
             ('Kupfer Cu', 0.0178), ('Aluminium Al', 0.0283),
             ('Eisen Fe', 0.100),   ('Konstantan', 0.500)],
    description='Material:', style={'description_width': '130px'},
    layout=widgets.Layout(width='320px')
)

R_leitung  = make_input('R [Ω]        =')
rho_inp    = make_input('ρ [Ω·mm²/m]  =', 'z.B. 0.0178 für Cu')
l_inp      = make_input('l [m]         =')
A_inp      = make_input('A [mm²]       =')

def set_rho(change):
    if change['new'] is not None:
        rho_inp.value = str(change['new'])
rho_select.observe(set_rho, names='value')

btn_leit = make_button()
out_leit = widgets.Output()

def calc_leit(_=None):
    out_leit.clear_output()
    with out_leit:
        try:
            R  = p(R_leitung.value)
            rho = p(rho_inp.value)
            l  = p(l_inp.value)
            A  = p(A_inp.value)
            vals = {'R': R, 'ρ': rho, 'l': l, 'A': A}
            none_fields = [k for k, v in vals.items() if v is None]
            if len(none_fields) == 0:
                print("❌ Ein Feld leer lassen!")
                return
            if len(none_fields) > 1:
                print("❌ Nur ein Feld leer lassen!")
                return
            miss = none_fields[0]
            # R = rho * l / A  (rho in Ω·mm²/m, l in m, A in mm²)
            if miss == 'R':
                res = rho * l / A
                print(f"✅ R = ρ·l/A = {fmt(res, 'Ω')}")
            elif miss == 'ρ':
                res = R * A / l
                print(f"✅ ρ = R·A/l = {res:.6g} Ω·mm²/m")
            elif miss == 'l':
                res = R * A / rho
                print(f"✅ l = R·A/ρ = {fmt(res, 'm')}")
            elif miss == 'A':
                res = rho * l / R
                print(f"✅ A = ρ·l/R = {res:.6g} mm²")
        except Exception as e:
            print("❌ Fehler:", e)

btn_leit.on_click(calc_leit)
for f in [R_leitung, rho_inp, l_inp, A_inp]:
    f.observe(calc_leit, names='value')

display(widgets.VBox([rho_select, R_leitung, rho_inp, l_inp, A_inp, btn_leit, out_leit]))

---
## 7. 🎨 Farbcode-Lesung (E12/E24 Widerstände)

Standard Farbcode (4-Ring):
```
Ring 1: Erster Ziffernwert
Ring 2: Zweiter Ziffernwert  
Ring 3: Multiplikator (10^x)
Ring 4: Toleranz
```

| Farbe | Ziffer | Multiplikator | Toleranz |
|-------|--------|---------------|----------|
| Schwarz | 0 | ×1 | – |
| Braun | 1 | ×10 | ±1% |
| Rot | 2 | ×100 | ±2% |
| Orange | 3 | ×1k | – |
| Gelb | 4 | ×10k | – |
| Grün | 5 | ×100k | ±0.5% |
| Blau | 6 | ×1M | ±0.25% |
| Violett | 7 | ×10M | ±0.1% |
| Grau | 8 | – | ±0.05% |
| Weiss | 9 | – | – |
| Gold | – | ×0.1 | ±5% |
| Silber | – | ×0.01 | ±10% |

In [ ]:
# ============================================================
# RECHNER 7: Widerstand-Farbcode Decoder (4-Ring)
# ============================================================
display(HTML("<h3>🔢 Rechner 7 – Farbcode Decoder (4-Ring)</h3>"))

COLORS = {
    'Schwarz': (0,  1,      None),
    'Braun':   (1,  10,     '±1%'),
    'Rot':     (2,  100,    '±2%'),
    'Orange':  (3,  1000,   None),
    'Gelb':    (4,  10000,  None),
    'Grün':    (5,  100000, '±0.5%'),
    'Blau':    (6,  1e6,    '±0.25%'),
    'Violett': (7,  1e7,    '±0.1%'),
    'Grau':    (8,  None,   '±0.05%'),
    'Weiss':   (9,  None,   None),
    'Gold':    (None, 0.1,  '±5%'),
    'Silber':  (None, 0.01, '±10%'),
}

color_opts_12 = [c for c in COLORS if COLORS[c][0] is not None]  # Rings 1+2
color_opts_3  = [c for c, (d, m, t) in COLORS.items() if m is not None]
color_opts_4  = [c for c, (d, m, t) in COLORS.items() if t is not None]

dd_style = {'description_width': '130px'}
dd_layout = widgets.Layout(width='320px')

ring1 = widgets.Dropdown(options=color_opts_12, description='Ring 1 (Ziffer 1):', style=dd_style, layout=dd_layout)
ring2 = widgets.Dropdown(options=color_opts_12, description='Ring 2 (Ziffer 2):', style=dd_style, layout=dd_layout)
ring3 = widgets.Dropdown(options=color_opts_3,  description='Ring 3 (Multiplik.):', style=dd_style, layout=dd_layout)
ring4 = widgets.Dropdown(options=color_opts_4,  description='Ring 4 (Toleranz):', style=dd_style, layout=dd_layout, value='Gold')

out_farb = widgets.Output()

def calc_farb(_=None):
    out_farb.clear_output()
    with out_farb:
        try:
            d1 = COLORS[ring1.value][0]
            d2 = COLORS[ring2.value][0]
            mult = COLORS[ring3.value][1]
            tol = COLORS[ring4.value][2]
            R = (d1 * 10 + d2) * mult
            print(f"✅ R = {fmt(R, 'Ω')}  {tol}")
            R_min = R * (1 - float(tol.strip('±%')) / 100)
            R_max = R * (1 + float(tol.strip('±%')) / 100)
            print(f"   Bereich: {fmt(R_min, 'Ω')} … {fmt(R_max, 'Ω')}")
        except Exception as e:
            print("❌ Fehler:", e)

for dd in [ring1, ring2, ring3, ring4]:
    dd.observe(calc_farb, names='value')

calc_farb()
display(widgets.VBox([ring1, ring2, ring3, ring4, out_farb]))

---
## 8. 🔋 Spannungsteiler (unbelastet)

$$U_2 = U_{ges} \cdot \frac{R_2}{R_1 + R_2}$$

Oder allgemein:
$$U_x = U_{ges} \cdot \frac{R_x}{R_{ges}}$$

> ⚠️ **Wichtig:** Diese Formel gilt nur für den **unbelasteten** Spannungsteiler (kein Strom wird abgenommen)!

In [ ]:
# ============================================================
# RECHNER 8: Spannungsteiler (unbelastet)
# ============================================================
display(HTML("<h3>🔢 Rechner 8 – Spannungsteiler (unbelastet)</h3>"
             "<i>Ein Feld leer lassen → wird berechnet</i>"))

Uges_st = make_input('U_ges [V] =')
R1_st   = make_input('R1 [Ω]   =')
R2_st   = make_input('R2 [Ω]   =')
U2_st   = make_input('U2 [V]   =')
btn_st  = make_button()
out_st  = widgets.Output()

def calc_st(_=None):
    out_st.clear_output()
    with out_st:
        try:
            Ug = p(Uges_st.value)
            R1 = p(R1_st.value)
            R2 = p(R2_st.value)
            U2 = p(U2_st.value)
            vals = {'U_ges': Ug, 'R1': R1, 'R2': R2, 'U2': U2}
            none_fields = [k for k, v in vals.items() if v is None]
            if len(none_fields) != 1:
                print("ℹ️ Genau ein Feld leer lassen!" if len(none_fields) > 1 else "❌ Ein Feld leer lassen!")
                return
            miss = none_fields[0]
            if miss == 'U2':
                res = Ug * R2 / (R1 + R2)
                print(f"✅ U2    = {fmt(res, 'V')}")
                print(f"   U1    = {fmt(Ug - res, 'V')}")
                I = Ug / (R1 + R2)
                print(f"   I     = {fmt(I, 'A')}")
            elif miss == 'U_ges':
                res = U2 * (R1 + R2) / R2
                print(f"✅ U_ges = {fmt(res, 'V')}")
            elif miss == 'R1':
                res = R2 * (Ug - U2) / U2
                print(f"✅ R1    = {fmt(res, 'Ω')}")
            elif miss == 'R2':
                res = R1 * U2 / (Ug - U2)
                print(f"✅ R2    = {fmt(res, 'Ω')}")
        except Exception as e:
            print("❌ Fehler:", e)

btn_st.on_click(calc_st)
for f in [Uges_st, R1_st, R2_st, U2_st]:
    f.observe(calc_st, names='value')

display(widgets.VBox([Uges_st, R1_st, R2_st, U2_st, btn_st, out_st]))

---
## 📋 Zusammenfassung: Wichtige Formeln

| Formel | Beschreibung |
|--------|-------------|
| $U = R \cdot I$ | Ohmsches Gesetz |
| $P = U \cdot I = I^2 R = U^2/R$ | Elektrische Leistung |
| $R_{ges} = R_1 + R_2 + \ldots$ | Reihenschaltung |
| $\frac{1}{R_{ges}} = \frac{1}{R_1} + \frac{1}{R_2} + \ldots$ | Parallelschaltung |
| $R_T = R_{20} \cdot [1 + \alpha \cdot (T-20)]$ | Temperaturabhängigkeit |
| $R = \rho \cdot l / A$ | Leitungswiderstand |
| $U_2 = U_{ges} \cdot R_2/(R_1+R_2)$ | Spannungsteiler |

---
*Elektroniker EFZ – Kapitel Widerstand | Jupyter Lab Zusammenfassung*